In [1]:
# =====================================================================
# DQA 智慧降噪 + 專家規則升級版 (完全對齊您的原始欄位)
# =====================================================================
!pip install openpyxl -q
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from google.colab import files

print("🤖 【第一步：請上傳您的 Excel 檔案】")
print("\n👉 1. 請上傳您的【歷史紀錄總檔案 (history_data.xlsx)】:")
uploaded_h = files.upload()
h_file = list(uploaded_h.keys())[0]

print("\n👉 2. 請上傳專案【待分類 BOM List (project_BOM_list.xlsx)】:")
uploaded_b = files.upload()
b_file = list(uploaded_b.keys())[0]

# =====================================================================
# 2. 讀取與智慧清洗 (多數決降噪)
# =====================================================================
print("\n🔄 【第二步：正在讀取並執行歷史資料『多數決降噪』...】")
df_history = pd.read_excel(h_file)
df_bin_list = pd.read_excel(b_file)

df_history = df_history.dropna(subset=['Description', 'Category', 'Subcategory'])

# 智慧多數決
df_cleaned = df_history.groupby('Description').agg({
    'Category': lambda x: x.value_counts().index[0],
    'Subcategory': lambda x: x.value_counts().index[0]
}).reset_index()

# =====================================================================
# 3. XGBoost 機器學習模型訓練
# =====================================================================
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=8000, analyzer='char_wb')
X_train = vectorizer.fit_transform(df_cleaned['Description'].astype(str).values)
X_predict = vectorizer.transform(df_bin_list['Description'].astype(str).values)

le_cat = LabelEncoder()
le_sub = LabelEncoder()
y_train_cat = le_cat.fit_transform(df_cleaned['Category'].astype(str).values)
y_train_sub = le_sub.fit_transform(df_cleaned['Subcategory'].astype(str).values)

print("\n🧠 【第三步：XGBoost AI 正在讀取歷史紀錄、學習分類邏輯...】")
model_cat = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.08, tree_method='hist', random_state=42)
model_sub = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.08, tree_method='hist', random_state=42)

model_cat.fit(X_train, y_train_cat)
model_sub.fit(X_train, y_train_sub)

# =====================================================================
# 4. AI 預測填入
# =====================================================================
print("\n🔮 【第四步：AI 預測與專家硬規則覆寫中...】")
pred_cat = model_cat.predict(X_predict)
pred_sub = model_sub.predict(X_predict)

prob_cat = np.max(model_cat.predict_proba(X_predict), axis=1) * 100
prob_sub = np.max(model_sub.predict_proba(X_predict), axis=1) * 100

# 先填入 AI 預測的結果
df_bin_list['Category'] = le_cat.inverse_transform(pred_cat)
df_bin_list['Subcategory'] = le_sub.inverse_transform(pred_sub)
df_bin_list['AI_Confidence_%'] = np.minimum(prob_cat, prob_sub).round(1)

# =====================================================================
# 🔥 🔥 🔥 核心防呆：強制執行 DQA 專家覆寫規則 🔥 🔥 🔥
# =====================================================================
# 只要 Description 裡面包含特定的關鍵字，不論歷史資料分得多爛，一律強制修正為標準答案！
for idx, row in df_bin_list.iterrows():
    desc_upper = str(row['Description']).upper()
    # 規則 1：只要有 BEAD (磁珠) 或 CHOKE (共模電感)，一律強制歸類為 Inductor ➔ Coil
    if 'BEAD' in desc_upper or 'CHOKE' in desc_upper:
        df_bin_list.at[idx, 'Category'] = 'Inductor'
        df_bin_list.at[idx, 'Subcategory'] = 'Coil'
        df_bin_list.at[idx, 'AI_Confidence_%'] = 100.0  # 強制標記為 100% 準確
    # 規則 2：如果未來有發現其他嚴重分錯的零件（例如某種特定 IC），可以比照辦理加在這裡

# =====================================================================
# 5. 導出並自動下載
# =====================================================================
output_name = "Project_Bin_List_AI_Fixed_100Percent.xlsx"
df_bin_list.to_excel(output_name, index=False)

print(f"\n🎉 【大功告成！】已經成功攔截並強制修正歷史人為錯誤！")
print(f"💾 重新下載極致完美版 Excel：{output_name}")
files.download(output_name)

🤖 【第一步：請上傳您的 Excel 檔案】

👉 1. 請上傳您的【歷史紀錄總檔案 (history_data.xlsx)】:


Saving history_data.xlsm to history_data.xlsm

👉 2. 請上傳專案【待分類 BOM List (project_BOM_list.xlsx)】:


Saving BDB-FP00050AA02_DB-FP0001-T7_BOM_分類_ok.xlsx to BDB-FP00050AA02_DB-FP0001-T7_BOM_分類_ok.xlsx

🔄 【第二步：正在讀取並執行歷史資料『多數決降噪』...】

🧠 【第三步：XGBoost AI 正在讀取歷史紀錄、學習分類邏輯...】

🔮 【第四步：AI 預測與專家硬規則覆寫中...】

🎉 【大功告成！】已經成功攔截並強制修正歷史人為錯誤！
💾 重新下載極致完美版 Excel：Project_Bin_List_AI_Fixed_100Percent.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>